# Projeto: Funções Clima Tempo

### Objetivo  
Este notebook implementa funções auxiliares para coletar, extrair e estruturar dados meteorológicos de forma padronizada do Clima Tempo. Ele permite gerar DataFrames de previsão para múltiplas cidades brasileiras, com foco em integração com pipelines de dados.

### Informações Técnicas

- **Tecnologias Utilizadas**: 
  - Apache Spark (PySpark)
  - Databricks Notebooks
 
---

### Histórico de Atualizações

| Data       | Responsável      | Versão | Descrição                                |
|------------|------------------|--------|--------------------------------------------|
| 14/05/2025 | Evelise Dias e Lucas Berta     |  1.0   | Criação de funções auxiliares para coleta de dados|
| 19/05/2025 | Evelise Dias   |  1.1   | Inclusão de documentação|


# Configurações Iniciais

In [0]:
%run ../utils_run

In [0]:
%run ./funcoes_config/

In [0]:
%run ./funcoes_comuns_motores

In [0]:
%run ../../../config/enum_clima_tempo

### Helpers (Funções Auxiliares)
Foram implementadas funções para modularizar e tornar o código de coleta mais legível e robusto.


In [0]:
def junta_uf_e_cidade(cidade: str, uf: str) -> str:
    """
    Concatena o nome da cidade e a sigla da UF no formato esperado pela URL ClimaTempo.

    Parameters:
        cidade (str): Nome da cidade.
        uf (str): Sigla da unidade federativa (UF).

    Returns:
        str: String no formato 'cidade-uf', onde a cidade é convertida para minúsculas sem espaços 
             e a UF também em minúsculas.
    """
    return cidade.lower().replace(" ", "") + "-" + uf.lower()

In [0]:
def extrair_texto(seletor) -> str:
    """
    Extrai o texto limpo de um objeto BeautifulSoup representando um seletor HTML.

    Parameters:
        seletor (Tag or None): Objeto BeautifulSoup (p.ex. resultado de select_one) que contém o elemento HTML.

    Returns:
        str or None: O texto limpo (com os espaços normalizados) extraído do seletor ou None se o seletor for inexistente.
    """
    if not seletor:
        return None
    return ' '.join(seletor.stripped_strings)

In [0]:
def obter_previsao_do_dia(secao: str) -> dict:
    """
    Realiza o parsing de uma seção HTML que representa um dia da previsão do tempo,
    retornando um dicionário estruturado com as informações meteorológicas relevantes.
    
    Os dados extraídos incluem:
        - Temperatura máxima
        - Temperatura mínima
        - Precipitação
        - Vento
        - Umidade mínima
        - Umidade máxima
    
    Os seletores CSS utilizados para a extração são definidos no enum 'ClimaTempo'.
    
    Parameters:
        secao (Tag): Objeto BeautifulSoup que contém a seção HTML de previsão de um dia.
    
    Returns:
        dict: Dicionário com as chaves definidas no enum ClimaTempo e os respectivos valores extraídos.
    """
    return {
        ClimaTempo.TEMPERATURA_MAXIMA.value: extrair_texto(secao.select_one(ClimaTempo.SEL_TEMPERATURA_MAXIMA.value)),
        ClimaTempo.TEMPERATURA_MINIMA.value: extrair_texto(secao.select_one(ClimaTempo.SEL_TEMPERATURA_MINIMA.value)),
        ClimaTempo.PRECIPITACAO.value: extrair_texto(secao.select_one(ClimaTempo.SEL_PRECIPITACAO.value)),
        ClimaTempo.VENTO.value: (
            extrair_texto(
                secao.select_one(ClimaTempo.SEL_VENTO_IDENTIFIER.value)
                .find_next("div", class_="-gray _flex _margin-l-5")
            ) if secao.select_one(ClimaTempo.SEL_VENTO_IDENTIFIER.value) else None
        ),
        ClimaTempo.UMIDADE_MINIMA.value: extrair_texto(secao.select_one(ClimaTempo.SEL_UMIDADE_MINIMA.value)),
        ClimaTempo.UMIDADE_MAXIMA.value: extrair_texto(secao.select_one(ClimaTempo.SEL_UMIDADE_MAXIMA.value)),
    }

In [0]:
def coletar_previsoes(nome_enum: str) -> list:
    """
    Realiza a coleta via web scraping das previsões meteorológicas para os próximos 15 dias 
    para todas as capitais descritas no enum especificado.
    
    Para cada cidade, a URL é construída utilizando o ID da cidade e o nome formatado (usando a função
    junta_uf_e_cidade). Em seguida, realiza-se uma requisição HTTP à URL de ClimaTempo. Caso a requisição seja 
    bem-sucedida, são extraídas as seções HTML correspondentes aos dias de previsão e, para cada uma, é realizado 
    o parsing por meio da função obter_previsao_do_dia. São adicionados dados extras, como cidade, estado e a data 
    da previsão (calculada com addDays).
    
    Parameters:
        nome_enum (str): Nome do enum (definido no módulo 'enum_configs') que contém os dados das cidades 
                         (ID e capital) para acessar as informações.
    
    Returns:
        list: Lista de dicionários contendo as informações estruturadas de previsão para cada dia.
    """
    enum_class = getattr(enum_configs, nome_enum)
    dados_cidades = []

    for uf, dados in enum_class.value.items():
        cidade = dados["capital"]
        cidade_id = dados["id"]
        cidade_formatada = junta_uf_e_cidade(cidade, uf)

        url = f"{ClimaTempo.URL_BASE.value}/{cidade_id}/{cidade_formatada}"

        # Fazendo a requisição HTTP
        try:
            response = requests.get(url, timeout=15)
            response.raise_for_status()
        except requests.RequestException as e:
            print(f"[ERRO] Falha ao coletar dados para {cidade}/{uf}: {e}")
            continue

        soup = BeautifulSoup(response.text, "html.parser")
        secoes = soup.select(f"{ClimaTempo.SEL_SEC_PREVISAO.value}")

        data_atual = datetime.now().strftime("%Y-%m-%d")

        for i, secao in enumerate(secoes):
            previsao = obter_previsao_do_dia(secao)
            previsao[f"{ClimaTempo.CIDADE.value}"] = cidade
            previsao[f"{ClimaTempo.ESTADO.value}"] = uf
            previsao[f"{ClimaTempo.DATA_PREVISAO.value}"] = addDays(data_atual, i)
            dados_cidades.append(previsao)

    return dados_cidades